## Lab Measurements

In [ ]:
import pandas            as pd
import miceforest        as mf
import matplotlib.pyplot as plt
import numpy             as np
import os
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
os.chdir(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805')

In [ ]:
df_lab = pd.read_parquet('lab_20240805.parquet')

In [ ]:
df_lab.studyId_0831 = df_lab.studyId_0831.astype(int) # Converto to int to have more clear view

In [ ]:
df_lab.columns # How many of those are relevant to us?

In [ ]:
units = df_lab.groupby('code_display_original').last().valueQuantity_code_original.to_list()

Let's remove uneuseful columns

In [ ]:
df_lab = df_lab.drop(['identifier_value', 'basedOn_ProcedureRequest_value',
                      'LabID', 'category3_display_original', 'category4_code_original',
                      'code_code_original','code2_code_original',
                      'code2_display_original', 'code3_display_original',
                      'interpretation_display',
                      'code3_system_original', 'code4_display_original', 'comment',
                      'status_display_original',
                      'valueQuantity_comparator', 'valueQuantity_unit',
                      'valueQuantity_unit_original', 'valueString',
                      'Practitioner_Rol', 'gender', 'organization_Organization_value',
                      'specialty_specialtyAGB_display', 'index_date'],axis='columns')

Storing the reference range of values per substance

In [ ]:
substances = pd.read_excel(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\excel\measurements_ranges.xlsx')
substances = substances.set_index('substance')


substances.loc['Kreatinine']['unit']

Pivoting on the chemical sub

In [ ]:
df_lab_pivoted = df_lab.pivot(columns='code_display_original', values='valueQuantity_value')
df_lab_pivoted

Adding useful columns

In [ ]:
df_lab_pivoted.insert(0, 'studyId_0831', df_lab.studyId_0831.values,allow_duplicates=True)
df_lab_pivoted['effectiveDateTime'] =  df_lab.effectiveDateTime

In [ ]:
df_lab_pivoted.columns

In [ ]:
ax = df_lab_pivoted[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
       'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
       'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',]]\
              .plot(kind='box',
                    subplots=True,
                    layout=[3,4],
                    figsize=[20,10],
                    )

for a,l in zip(ax,units):
    a.set_ylabel(r'${}$'.format(l))


In [ ]:

myColors = ('#0D47A1', '#E3F2FD')
mywhite = ('#FFFFFF','#FFFFFF')
cmap = LinearSegmentedColormap.from_list('Custom', myColors, len(myColors))
cmap2 = LinearSegmentedColormap.from_list('Custom', mywhite, len(mywhite))
fig = plt.figure(figsize=[12,8])
ax = fig.subplot_mosaic('''A;B''',sharex=True,height_ratios=[0.1,1],)

fig.add_axes(sns.heatmap(df_lab_pivoted[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
                                'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
                                'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',
                                ]].isna().sum().to_numpy().reshape(1,12),
                 yticklabels=['Total'],
                 ax=ax['A'],
                 cmap=cmap2,
                 cbar=True,
                 annot=True,
                 annot_kws={'rotation':0},
                 fmt='d',
                 cbar_kws={'alpha':0}
                 ))
fig.add_axes(
    sns.heatmap(df_lab_pivoted[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
                                'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
                                'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',
                                ]].isna(),
                 yticklabels='', 
                 cmap=cmap,
                 ax=ax['B']
                 ))
fig.subplots_adjust(hspace=0)
fig.subplots_adjust()
ax['B'].set_xlabel('Hematochemical Substance')
cb = ax['B'].collections[0].colorbar
cb.set_ticks([0,1])
cb.set_ticklabels(['Present', 'Missing'])
ax['B'].set_xlabel('Hematochemical Substance')

cb = ax['A'].collections[0].colorbar
cb.set_ticks([])
cb.set_ticklabels('')
cb.set_alpha(0)

## Patient Level exploration

---

Group by patients, convert to list to have the complete timeseries

In [ ]:
df_lab_timeseries = df_lab_pivoted.groupby('studyId_0831').agg(lambda x: x.to_list()).reset_index()
df_lab_timeseries

In [ ]:

fig = plt.figure(figsize=[50,8])
ax = fig.subplots()
patient = 1139
# np.random.choice(['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
c = 'Kreatinine'
    #                  'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
    #                  'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol'])
ax.plot(#df_lab_timeseries.iloc[patient]['effectiveDateTime'],
                   df_lab_timeseries.iloc[patient][c],
                   '--o',
                   c = 'DarkBlue',
                )
#ax.set_xticks(df_lab_timeseries.loc[patient]['effectiveDateTime'])
ax.tick_params(axis='x', rotation=90)
ax.set_xlabel('Time')
ax.set_ylabel(f'{c}')

---

Impute values based on some nice MICE Forest algorithm:
- Pivoted on <code>code_display_original</code>
- Run MICE Forest

In [ ]:
ker = mf.ImputationKernel(
    df_lab_pivoted[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
                    'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
                    'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',
                    ]].reset_index(drop=True),
    num_datasets=2,
    save_all_iterations_data=True,
    random_state=42)

ker.mice(iterations = 5,
         n_estimators = 100,
         boosting = 'gbdt',
         min_sum_hessian_in_leaf = 0.01)

In [ ]:

df_lab_imputed = ker.complete_data()
df_lab_imputed

In [ ]:
df_lab_imputed = pd.concat([df_lab_imputed, df_lab_pivoted[['effectiveDateTime', 'studyId_0831']]],axis='columns')
df_lab_imputed

In [ ]:
df_lab_imputed = df_lab_imputed.set_index('studyId_0831').sort_index()
df_lab_imputed

In [ ]:
tmp =  df_lab_imputed.groupby('studyId_0831')\
            .agg(lambda x: x.to_list()).reset_index()
fig = plt.figure(figsize=[50, 8])
ax = fig.subplots()
patient = 1139
c = np.random.choice(['Kreatinine'])
ax.plot(#tmp.iloc[patient]['effectiveDateTime'],
        tmp.iloc[patient][c],
        '--o',
        c='DarkBlue',
        )
#ax.set_xticks(tmp.iloc[patient]['effectiveDateTime'])
ax.tick_params(axis='x', rotation=90)
ax.set_xlabel('Time')
ax.set_ylabel(f'{c}')

In [ ]:
pd.DataFrame(df_lab_timeseries.reset_index().iloc[patient][c],columns='Kreatinine')

In [ ]:
tmp = df_lab_imputed.groupby('studyId_0831')\
    .agg(lambda x: x.to_list()).reset_index()
fig = plt.figure(figsize=[6,6])
ax = fig.subplots()
patient = 1139
c = np.random.choice(['Kreatinine'])
ax.boxplot(x=[tmp.iloc[patient][c], df_lab_timeseries.reset_index().iloc[patient][c]], tick_labels=['Imputed', 'original'])
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel(r'{}'.format(c) + ' ' + r'$\left[\frac{\mu\;\text{mol}}{L}\right]$')

In [ ]:

myColors = ('#0D47A1', '#E3F2FD')
mywhite = ('#FFFFFF', '#FFFFFF')
cmap = LinearSegmentedColormap.from_list('Custom', myColors, len(myColors))
cmap2 = LinearSegmentedColormap.from_list('Custom', mywhite, len(mywhite))
fig = plt.figure(figsize=[12, 8])
ax = fig.subplot_mosaic('''A;B''', sharex=True, height_ratios=[0.1, 1],)

fig.add_axes(sns.heatmap(df_lab_imputed[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
                                         'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
                                         'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',
                                         ]].isna().sum().to_numpy().reshape(1, 12),
                         yticklabels=['Total'],
                         ax=ax['A'],
                         cmap=cmap2,
                         cbar=True,
                         annot=True,
                         annot_kws={'rotation': 0},
                         fmt='d',
                         cbar_kws={'alpha': 0}
                         ))
fig.add_axes(
    sns.heatmap(df_lab_imputed[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
                                'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
                                'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',
                                ]].isna(),
                yticklabels='',
                cmap=cmap,
                ax=ax['B']
                ))
fig.subplots_adjust(hspace=0)
fig.subplots_adjust()
ax['B'].set_xlabel('Hematochemical Substance')
cb = ax['B'].collections[0].colorbar
cb.set_ticks([0, 1])
cb.set_ticklabels(['Present', 'Missing'])
ax['B'].set_xlabel('Hematochemical Substance')

cb = ax['A'].collections[0].colorbar
cb.set_ticks([])
cb.set_ticklabels('')
cb.set_alpha(0)

In [ ]:
#ker.plot_imputed_distributions()

Try to aggregate values with using weghted mean:<br>
Weights are evaluated based on the number of rows...<br>
NaN values are filled with 0s

In [ ]:
def wavg_last_n(x : pd.Series):
  try:
    weights = np.arange(start=1,stop=x.count()+1)/x.count()
    return np.average(x,weights=weights)
  except:
     return x.tail(1)

In [ ]:
df_lab_imputed.groupby('studyId_0831')\
                    .agg(lambda x: wavg_last_n(x))

In [ ]:
ax = df_lab_imputed[['CRP', 'Cholesterol', 'HDL Cholesterol', 'HbA1c',
                     'Hemoglobine', 'Kalium', 'Kreatinine', 'LDL Cholesterol', 'Natrium',
                     'Triglyceriden', 'eGFR (CKD-EPI)', 'non-HDL cholesterol',]]\
    .plot(kind='box',
          subplots=True,
          layout=[3, 4],
          figsize=[20, 10],
          )

for a, l in zip(ax, units):
    a.set_ylabel(r'${}$'.format(l))